# SentimentIQ — Model Training
Train a Logistic Regression sentiment classifier using TF-IDF features on the NLTK movie reviews + custom dataset.

In [41]:
import subprocess
subprocess.check_call(["pip", "install", "pandas", "numpy", "scikit-learn", "nltk", "joblib"])
print("Done!")

Done!


In [42]:
import pandas as pd
import numpy as np
import re
import os
import joblib
import nltk
from nltk.corpus import stopwords, movie_reviews
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

nltk.download('movie_reviews')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
print('Libraries loaded!')

Libraries loaded!


[nltk_data] Downloading package movie_reviews to C:\Users\Vasvi
[nltk_data]     Shukla\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Vasvi
[nltk_data]     Shukla\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Vasvi
[nltk_data]     Shukla\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Vasvi
[nltk_data]     Shukla\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [43]:
# ── Load NLTK Movie Reviews Dataset ─────────────────────────
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

texts, labels = [], []
for words, category in documents:
    texts.append(' '.join(words))
    labels.append(2 if category == 'pos' else 0)  # 2=Positive, 0=Negative

# Add neutral examples (midpoint reviews)
neutral_examples = [
    'The product is okay, nothing special but does the job.',
    'It was an average experience, neither good nor bad.',
    'The movie was fine. Some parts were interesting.',
    'Decent quality for the price. Could be better.',
    'Not sure how I feel about this. It was just alright.',
    'The service was standard. No complaints but nothing impressive.',
    'It is what it is. A fair product overall.',
    'Mixed feelings. Has both pros and cons.',
    'An acceptable experience. Would not strongly recommend or avoid.',
    'The results were moderate. Expected more but not disappointed.',
] * 50  # Duplicate to balance dataset

texts.extend(neutral_examples)
labels.extend([1] * len(neutral_examples))  # 1=Neutral

df = pd.DataFrame({'text': texts, 'label': labels})
print(f'Dataset size: {len(df)}')
print(df['label'].value_counts().rename({0:'Negative', 1:'Neutral', 2:'Positive'}))

Dataset size: 2500
label
Negative    1000
Positive    1000
Neutral      500
Name: count, dtype: int64


In [44]:
# ── Preprocessing ─────────────────────────────────────────────
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['cleaned'] = df['text'].apply(preprocess)
print('Preprocessing done!')
print(df[['text', 'cleaned']].head(3))

Preprocessing done!
                                                text  \
0  plot : two teen couples go to a church party ,...   
1  the happy bastard ' s quick movie review damn ...   
2  it is movies like these that make a jaded movi...   

                                             cleaned  
0  plot two teen couple go church party drink dri...  
1  happy bastard quick movie review damn yk bug g...  
2  movie like make jaded movie viewer thankful in...  


In [45]:
# ── Train/Test Split ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

Train: 2000 | Test: 500


In [46]:
# ── TF-IDF Vectorization ──────────────────────────────────────
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(f'Feature matrix shape: {X_train_vec.shape}')

Feature matrix shape: (2000, 10000)


In [47]:
# ── Train Logistic Regression Model ───────────────────────────
model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
model.fit(X_train_vec, y_train)
print('Model trained!')

Model trained!


In [48]:
# ── Evaluation ─────────────────────────────────────────────────
y_pred = model.predict(X_test_vec)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

Accuracy: 0.8760

Classification Report:
              precision    recall  f1-score   support

    Negative       0.86      0.82      0.84       200
     Neutral       1.00      1.00      1.00       100
    Positive       0.83      0.86      0.85       200

    accuracy                           0.88       500
   macro avg       0.90      0.90      0.90       500
weighted avg       0.88      0.88      0.88       500



In [49]:
# ── Save Model ─────────────────────────────────────────────────
os.makedirs('../model', exist_ok=True)
joblib.dump(model, '../model/sentiment_model.pkl')
joblib.dump(vectorizer, '../model/tfidf_vectorizer.pkl')
print('Model and vectorizer saved to /model!')

Model and vectorizer saved to /model!


In [50]:
# ── Test Predictions ───────────────────────────────────────────
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
test_texts = [
    'This product is absolutely amazing, I love it!',
    'Terrible experience. Would not recommend to anyone.',
    'It was okay, nothing special really.',
    'The quality exceeded my expectations. Very happy!',
    'Not great, not terrible. Just average.'
]
for t in test_texts:
    cleaned = preprocess(t)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = max(model.predict_proba(vec)[0]) * 100
    print(f'{label_map[pred]} ({prob:.1f}%) | {t}')

Positive (44.6%) | This product is absolutely amazing, I love it!
Neutral (43.9%) | Terrible experience. Would not recommend to anyone.
Neutral (68.6%) | It was okay, nothing special really.
Positive (50.2%) | The quality exceeded my expectations. Very happy!
Positive (45.2%) | Not great, not terrible. Just average.
